# LeetCode 1272 — Remove Interval bằng Python

Link đề: https://leetcode.com/problems/remove-interval/ · Độ khó: **Medium** · Chủ đề: Array · 🔒 **Premium**

Skeleton LeetCode (theo lời giải Python trên doocs):

```python
class Solution:
    def removeInterval(
        self, intervals: list[list[int]], toBeRemoved: list[int]
    ) -> list[list[int]]:
        ...
```

> **Ghi chú Premium:** đề bài trong notebook được viết lại theo hiểu biết về bài (tham khảo https://leetcode.doocs.org/en/lc/1272/). Nếu có tài khoản Premium, hãy đối chiếu lại đề gốc.

## 1. Mục tiêu học

1. Làm việc chính xác với **khoảng nửa mở** `[a, b)`: điểm đầu thuộc, điểm cuối không thuộc; biết khi nào hai khoảng giao nhau.
2. Phân loại quan hệ giữa một khoảng và khoảng bị xoá: **rời nhau / bị phủ hết / bị cắt trái / bị cắt phải / bị chẻ đôi**.
3. Đi từ **rời rạc hoá từng đơn vị** (brute force) → **phân trường hợp** → **công thức giao `min`/`max`** → **tìm kiếm nhị phân vùng bị ảnh hưởng** → **sweep line tổng quát**.
4. Viết bộ test có **oracle** (brute force trên điểm nguyên) và **hàm kiểm tra hợp lệ** (đã sắp, rời nhau, không rỗng).
5. Tránh lỗi: nhầm sang khoảng đóng, sinh khoảng rỗng `[x, x]`, dùng `elif` làm mất mảnh bên phải, quên trường hợp khoảng bao trùm.

## 2. Đề bài & ví dụ

Một tập số thực có thể biểu diễn bằng hợp của các khoảng **rời nhau**, mỗi khoảng dạng `[a, b)` (số `x` thuộc khoảng nếu `a <= x < b`).

Cho danh sách `intervals` **đã sắp xếp, rời nhau**, `intervals[i] = [a_i, b_i]` biểu diễn `[a_i, b_i)`, và một khoảng `toBeRemoved`. Trả về tập số thực sau khi **xoá** `toBeRemoved` khỏi `intervals` — tức mọi `x` thuộc `intervals` nhưng không thuộc `toBeRemoved` — dưới dạng danh sách khoảng **đã sắp xếp, rời nhau**.

**Ví dụ 1:** `intervals = [[0,2],[3,4],[5,7]]`, `toBeRemoved = [1,6]` → `[[0,1],[6,7]]`.

**Ví dụ 2:** `intervals = [[0,5]]`, `toBeRemoved = [2,3]` → `[[0,2],[3,5]]`.

**Ví dụ 3:** `intervals = [[-5,-4],[-3,-2],[1,2],[3,5],[8,9]]`, `toBeRemoved = [-1,4]` → `[[-5,-4],[-3,-2],[4,5],[8,9]]`.

**Ràng buộc:** `1 <= len(intervals) <= 10^4`; `-10^9 <= a_i < b_i <= 10^9`.

In [1]:
import bisect
import random
import time
from typing import List

examples = [
    ([[0, 2], [3, 4], [5, 7]], [1, 6], [[0, 1], [6, 7]]),
    ([[0, 5]], [2, 3], [[0, 2], [3, 5]]),
    ([[-5, -4], [-3, -2], [1, 2], [3, 5], [8, 9]], [-1, 4], [[-5, -4], [-3, -2], [4, 5], [8, 9]]),
]

def draw(intervals, removed=None, lo=None, hi=None):
    """Vẽ trục số: '=' thuộc tập, 'x' bị xoá, '.' trống (mỗi ký tự là một đoạn [t, t+1))."""
    pts = [p for iv in intervals for p in iv] + (list(removed) if removed else [])
    lo = min(pts) if lo is None else lo
    hi = max(pts) if hi is None else hi
    line = ""
    for t in range(lo, hi):
        inside = any(a <= t < b for a, b in intervals)
        rem = removed is not None and removed[0] <= t < removed[1]
        line += ("x" if rem else "=") if inside else ("-" if rem else ".")
    return f"{lo:>4} |{line}| {hi}"

for iv, rem, want in examples:
    print("trước:", draw(iv, rem))
    lo = min(p for x in iv for p in x); hi = max(p for x in iv for p in x)
    print("sau  :", draw(want, None, lo, hi), "\n")

trước:    0 |=x-x-x=| 7
sau  :    0 |=.....=| 7 

trước:    0 |==x==| 5
sau  :    0 |==.==| 5 

trước:   -5 |=.=.--x-x=...=| 9
sau  :   -5 |=.=......=...=| 9 



## 3. Kiến thức nền

### 3.1 Khoảng nửa mở `[a, b)`

- Độ dài là `b − a`; khoảng **rỗng** khi `a >= b`.
- `[a, b)` và `[c, d)` **giao nhau** ⇔ `a < d and c < b` (tức `max(a, c) < min(b, d)`).
- Hai khoảng "chạm" nhau `[0, 2)` và `[2, 5)` **không** giao (điểm 2 không thuộc khoảng đầu).

In [2]:
def overlap(p, q):
    (a, b), (c, d) = p, q
    return a < d and c < b

print(overlap([0, 2], [1, 6]), overlap([0, 2], [2, 5]), overlap([3, 4], [1, 6]), overlap([8, 9], [-1, 4]))

True False True False


### 3.2 Giao và hiệu của hai khoảng

- Giao: `[max(a, c), min(b, d))` (rỗng nếu `max >= min`).
- Hiệu `[a, b) \ [x, y)` có tối đa **hai mảnh**: mảnh trái `[a, min(b, x))` và mảnh phải `[max(a, y), b)`; mảnh nào rỗng thì bỏ.

In [3]:
def diff(a, b, x, y):
    pieces = []
    if a < min(b, x):
        pieces.append([a, min(b, x)])
    if max(a, y) < b:
        pieces.append([max(a, y), b])
    return pieces

for a, b in [(0, 2), (3, 4), (5, 7), (0, 10), (8, 9)]:
    print(f"[{a},{b}) \\ [1,6) = {diff(a, b, 1, 6)}")

[0,2) \ [1,6) = [[0, 1]]
[3,4) \ [1,6) = []
[5,7) \ [1,6) = [[6, 7]]
[0,10) \ [1,6) = [[0, 1], [6, 10]]
[8,9) \ [1,6) = [[8, 9]]


### 3.3 Unpack trong vòng lặp

`for a, b in intervals:` tách mỗi cặp thành hai biến; `x, y = toBeRemoved` tương tự. Giúp code đọc như công thức toán.

In [4]:
x, y = [1, 6]
for a, b in [[0, 2], [3, 4]]:
    print(f"a={a}, b={b}, x={x}, y={y}")

a=0, b=2, x=1, y=6
a=3, b=4, x=1, y=6


### 3.4 `bisect` trên danh sách đầu mút

Vì `intervals` đã sắp và rời nhau, cả dãy điểm đầu `starts` lẫn dãy điểm cuối `ends` đều tăng. `bisect_right(ends, x)` = số khoảng kết thúc `<= x` (nằm hẳn bên trái `x`).

In [5]:
iv = examples[2][0]
ends = [b for _, b in iv]
starts = [a for a, _ in iv]
print("ends =", ends, "| starts =", starts)
print("số khoảng nằm hẳn bên trái x=-1:", bisect.bisect_right(ends, -1))
print("số khoảng bắt đầu trước y=4:", bisect.bisect_left(starts, 4))

ends = [-4, -2, 2, 5, 9] | starts = [-5, -3, 1, 3, 8]
số khoảng nằm hẳn bên trái x=-1: 2
số khoảng bắt đầu trước y=4: 4


### 3.5 Hàm kiểm tra tính hợp lệ của kết quả

Kết quả phải: (1) mỗi khoảng không rỗng `a < b`; (2) sắp xếp và rời nhau `b_i <= a_{i+1}`. Viết sẵn hàm này để dùng trong test.

In [6]:
def is_valid(ans):
    return all(a < b for a, b in ans) and all(ans[i][1] <= ans[i + 1][0] for i in range(len(ans) - 1))

print(is_valid([[0, 1], [6, 7]]), is_valid([[0, 1], [1, 1]]), is_valid([[0, 3], [2, 5]]))

True False False


## 4. Ý tưởng nền tảng / trực giác

Tính tay **Ví dụ 1**: `intervals = [0,2), [3,4), [5,7)`, xoá `[1,6)`.

| Khoảng | Quan hệ với `[1,6)` | Phần còn lại |
|---|---|---|
| `[0,2)` | bị cắt phía phải (đè lên `[1,2)`) | `[0,1)` |
| `[3,4)` | nằm trọn trong vùng xoá | (không còn gì) |
| `[5,7)` | bị cắt phía trái (đè lên `[5,6)`) | `[6,7)` |

Kết quả `[[0,1],[6,7]]`.

**Ví dụ 2:** `[0,5)` **bao trùm** `[2,3)` ⇒ bị chẻ đôi thành `[0,2)` và `[3,5)`.

Mỗi khoảng xử lý **độc lập**, và vì các khoảng đã sắp + rời nhau, các mảnh sinh ra theo thứ tự cũng đã sắp + rời nhau.

In [7]:
x, y = 1, 6
for a, b in [[0, 2], [3, 4], [5, 7]]:
    if b <= x or a >= y:
        kind = "rời nhau"
    elif x <= a and b <= y:
        kind = "bị phủ hết"
    elif a < x and b > y:
        kind = "bị chẻ đôi"
    elif a < x:
        kind = "bị cắt phía phải"
    else:
        kind = "bị cắt phía trái"
    print(f"[{a},{b}): {kind:17} -> {diff(a, b, x, y)}")

[0,2): bị cắt phía phải  -> [[0, 1]]
[3,4): bị phủ hết        -> []
[5,7): bị cắt phía trái  -> [[6, 7]]


## 5. Cách 1 — Brute Force / baseline: rời rạc hoá theo đơn vị

Mọi đầu mút là **số nguyên**, nên tập số thực là hợp của các ô đơn vị `[t, t+1)` với `t` nguyên. Brute force:
1. Duyệt mọi `t` từ đầu mút nhỏ nhất đến lớn nhất.
2. Giữ ô `[t, t+1)` nếu nó thuộc một khoảng nào đó và **không** thuộc `toBeRemoved`.
3. Gộp các ô liên tiếp thành khoảng.

Chỉ dùng được khi toạ độ nhỏ (ràng buộc thật là `±10^9`!) — nhưng là **oracle** tuyệt vời cho test.

In [8]:
def remove_brute(intervals, toBeRemoved):
    x, y = toBeRemoved
    lo = intervals[0][0]
    hi = intervals[-1][1]
    ans = []
    for t in range(lo, hi):
        keep = any(a <= t < b for a, b in intervals) and not (x <= t < y)
        if keep:
            if ans and ans[-1][1] == t:
                ans[-1][1] = t + 1          # nối tiếp ô trước
            else:
                ans.append([t, t + 1])
    return ans

for iv, rem, want in examples:
    print(remove_brute(iv, rem), "| mong đợi", want)

[[0, 1], [6, 7]] | mong đợi [[0, 1], [6, 7]]
[[0, 2], [3, 5]] | mong đợi [[0, 2], [3, 5]]
[[-5, -4], [-3, -2], [4, 5], [8, 9]] | mong đợi [[-5, -4], [-3, -2], [4, 5], [8, 9]]


Lưu ý nhỏ: brute force **gộp** các khoảng chạm nhau (ví dụ `[0,2),[2,3)` thành `[0,3)`). Đề cho `intervals` rời nhau nhưng **có thể chạm nhau** không thì đề không nói rõ; lời giải chuẩn giữ nguyên ranh giới. Vì vậy khi so sánh, ta so **tập điểm** (sau khi chuẩn hoá bằng cách gộp các khoảng chạm nhau) — xem hàm `normalize` ở mục 12.

**Độ phức tạp:** `O(R · n)` với `R = hi − lo` (có thể tới `2·10^9`) — không khả thi với ràng buộc thật.

**Nút thắt:** ta làm việc trên từng điểm, trong khi mỗi khoảng chỉ có vài trường hợp quan hệ với vùng xoá — xử lý **cả khoảng một lúc**.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight:** với mỗi khoảng `[a, b)`, phần còn lại sau khi xoá `[x, y)` là:

> mảnh trái `[a, min(b, x))` (nếu không rỗng) + mảnh phải `[max(a, y), b)` (nếu không rỗng).

Công thức này **tự động** bao hết 5 trường hợp (rời nhau trái, rời nhau phải, bị phủ, bị cắt một phía, bị chẻ đôi).

**Quy tắc "nếu… thì…" (phân trường hợp, lời giải doocs):**
- Nếu `a >= y` hoặc `b <= x` ⇒ rời nhau ⇒ giữ nguyên `[a, b]`.
- Ngược lại (có giao):
  - nếu `a < x` ⇒ thêm mảnh trái `[a, x]`;
  - nếu `b > y` ⇒ thêm mảnh phải `[y, b]` (hai `if` **độc lập** — có thể thêm cả hai).

**Invariant khi duyệt trái → phải:** `ans` luôn sắp xếp và rời nhau, vì mỗi mảnh nằm trong khoảng gốc của nó, các khoảng gốc rời nhau và tăng dần, và trong cùng một khoảng mảnh trái `< x <= y <=` mảnh phải.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Phân trường hợp (lời giải doocs)

In [9]:
def remove_cases(intervals, toBeRemoved):
    x, y = toBeRemoved
    ans = []
    for a, b in intervals:
        if a >= y or b <= x:            # rời nhau
            ans.append([a, b])
        else:                            # có giao
            if a < x:
                ans.append([a, x])       # mảnh trái
            if b > y:
                ans.append([y, b])       # mảnh phải
    return ans

for iv, rem, want in examples:
    print(remove_cases(iv, rem), "| mong đợi", want)

[[0, 1], [6, 7]] | mong đợi [[0, 1], [6, 7]]
[[0, 2], [3, 5]] | mong đợi [[0, 2], [3, 5]]
[[-5, -4], [-3, -2], [4, 5], [8, 9]] | mong đợi [[-5, -4], [-3, -2], [4, 5], [8, 9]]


Bản nộp LeetCode (đúng như doocs):

In [10]:
class SolutionCases:
    def removeInterval(
        self, intervals: list[list[int]], toBeRemoved: list[int]
    ) -> list[list[int]]:
        x, y = toBeRemoved
        ans = []
        for a, b in intervals:
            if a >= y or b <= x:
                ans.append([a, b])
            else:
                if a < x:
                    ans.append([a, x])
                if b > y:
                    ans.append([y, b])
        return ans

print([SolutionCases().removeInterval(iv, r) == w for iv, r, w in examples])

[True, True, True]


**Độ phức tạp Cách 2:** **`O(n)` thời gian**, `O(1)` bộ nhớ phụ (không tính kết quả, tối đa `n + 1` khoảng).

### 7.2 Cách 3 — Công thức giao `min` / `max` (không cần nhánh "rời nhau")

Áp dụng thẳng công thức hiệu ở mục 3.2 cho mọi khoảng. Khi khoảng nằm hẳn bên trái `x`: `min(b, x) = b` ⇒ mảnh trái là cả khoảng, mảnh phải `[max(a, y), b)` rỗng. Khi nằm hẳn bên phải `y`: mảnh trái rỗng, mảnh phải là cả khoảng.

In [11]:
def remove_minmax(intervals, toBeRemoved):
    x, y = toBeRemoved
    ans = []
    for a, b in intervals:
        left_end = min(b, x)
        if a < left_end:
            ans.append([a, left_end])
        right_start = max(a, y)
        if right_start < b:
            ans.append([right_start, b])
    return ans

for iv, rem, want in examples:
    print(remove_minmax(iv, rem), "| mong đợi", want)

[[0, 1], [6, 7]] | mong đợi [[0, 1], [6, 7]]
[[0, 2], [3, 5]] | mong đợi [[0, 2], [3, 5]]
[[-5, -4], [-3, -2], [4, 5], [8, 9]] | mong đợi [[-5, -4], [-3, -2], [4, 5], [8, 9]]


**Độ phức tạp Cách 3:** `O(n)` thời gian, `O(1)` bộ nhớ phụ. Ưu điểm: không phải nhớ điều kiện "rời nhau", ít chỗ sai dấu `<`/`<=`.

### 7.3 Cách 4 — Tìm kiếm nhị phân vùng bị ảnh hưởng

Vì `intervals` đã sắp và rời nhau:
- `i = bisect_right(ends, x)`: các khoảng `0..i-1` kết thúc `<= x` ⇒ giữ nguyên.
- `j = bisect_left(starts, y)`: các khoảng `j..n-1` bắt đầu `>= y` ⇒ giữ nguyên.
- Chỉ các khoảng `i..j-1` giao với vùng xoá; chỉ khoảng `i` có thể có mảnh trái, chỉ khoảng `j−1` có thể có mảnh phải.

Tìm vùng ảnh hưởng `O(log n)`; nếu được trả về "view"/sửa tại chỗ thì phần việc thực sự chỉ `O(log n + số khoảng bị ảnh hưởng)`. Khi phải tạo danh sách mới thì vẫn `O(n)` do sao chép.

In [12]:
def remove_bisect(intervals, toBeRemoved):
    x, y = toBeRemoved
    starts = [a for a, _ in intervals]
    ends = [b for _, b in intervals]
    i = bisect.bisect_right(ends, x)        # khoảng đầu tiên có end > x
    j = bisect.bisect_left(starts, y)       # khoảng đầu tiên có start >= y
    middle = []
    if i < j:                               # có khoảng bị ảnh hưởng
        a_i = intervals[i][0]
        b_last = intervals[j - 1][1]
        if a_i < x:
            middle.append([a_i, x])
        if b_last > y:
            middle.append([y, b_last])
    return [iv[:] for iv in intervals[:i]] + middle + [iv[:] for iv in intervals[j:]]

for iv, rem, want in examples:
    print(remove_bisect(iv, rem), "| mong đợi", want)

[[0, 1], [6, 7]] | mong đợi [[0, 1], [6, 7]]
[[0, 2], [3, 5]] | mong đợi [[0, 2], [3, 5]]
[[-5, -4], [-3, -2], [4, 5], [8, 9]] | mong đợi [[-5, -4], [-3, -2], [4, 5], [8, 9]]


**Độ phức tạp Cách 4:** dựng `starts`/`ends` `O(n)` (có thể tránh nếu dùng `key=` của `bisect` trong Python ≥ 3.10), hai lần `bisect` `O(log n)`, ghép kết quả `O(n)`.

### 7.4 Cách 5 — Sweep line (quét điểm) tổng quát

Gán trọng số: mỗi khoảng `intervals` cộng `+1` trên `[a, b)`; vùng xoá cộng `−INF` (hoặc dùng hai bộ đếm). Quét các đầu mút đã sắp xếp; đoạn giữa hai đầu mút liên tiếp được giữ nếu "đang trong `intervals`" và "không trong vùng xoá". Cách này tổng quát cho **nhiều** vùng xoá / nhiều phép toán tập hợp (hợp, giao, hiệu).

In [13]:
def remove_sweep(intervals, toBeRemoved):
    events = []
    for a, b in intervals:
        events.append((a, 0, +1)); events.append((b, 0, -1))
    x, y = toBeRemoved
    events.append((x, 1, +1)); events.append((y, 1, -1))
    events.sort()
    cover = [0, 0]                 # cover[0]: số khoảng intervals đang mở; cover[1]: vùng xoá đang mở
    ans = []
    prev = None
    for pos, kind, delta in events:
        if prev is not None and pos > prev and cover[0] > 0 and cover[1] == 0:
            if ans and ans[-1][1] == prev:
                ans[-1][1] = pos   # gộp đoạn liền kề
            else:
                ans.append([prev, pos])
        cover[kind] += delta
        prev = pos
    return ans

for iv, rem, want in examples:
    print(remove_sweep(iv, rem), "| mong đợi", want)

[[0, 1], [6, 7]] | mong đợi [[0, 1], [6, 7]]
[[0, 2], [3, 5]] | mong đợi [[0, 2], [3, 5]]
[[-5, -4], [-3, -2], [4, 5], [8, 9]] | mong đợi [[-5, -4], [-3, -2], [4, 5], [8, 9]]


**Độ phức tạp Cách 5:** `O(n log n)` (sắp xếp sự kiện), `O(n)` bộ nhớ. Chậm hơn Cách 2/3 cho bài này, nhưng là "dao đa năng" cho các bài phép toán trên khoảng. Giống brute force, cách này **gộp** các khoảng chạm nhau.

## 8. TRACE từng bước

Trace **Cách 2** với Ví dụ 3: `intervals = [[-5,-4],[-3,-2],[1,2],[3,5],[8,9]]`, `toBeRemoved = [-1,4]`:

| `[a, b)` | `a >= y` hoặc `b <= x`? | `a < x`? | `b > y`? | thêm vào `ans` |
|---|---|---|---|---|
| `[-5,-4)` | `b=-4 <= -1` ✔ rời | — | — | `[-5,-4]` |
| `[-3,-2)` | `b=-2 <= -1` ✔ rời | — | — | `[-3,-2]` |
| `[1,2)` | ✘ giao | `1 < -1` ✘ | `2 > 4` ✘ | (không) |
| `[3,5)` | ✘ giao | `3 < -1` ✘ | `5 > 4` ✔ | `[4,5]` |
| `[8,9)` | `a=8 >= 4` ✔ rời | — | — | `[8,9]` |

In [14]:
def trace_cases(intervals, toBeRemoved):
    x, y = toBeRemoved
    ans = []
    print(f"xoá [{x},{y})")
    for a, b in intervals:
        if a >= y or b <= x:
            ans.append([a, b])
            why = "rời nhau -> giữ nguyên"
        else:
            parts = []
            if a < x:
                ans.append([a, x]); parts.append(f"mảnh trái [{a},{x})")
            if b > y:
                ans.append([y, b]); parts.append(f"mảnh phải [{y},{b})")
            why = "giao -> " + (", ".join(parts) if parts else "bị xoá hết")
        print(f"  [{a:>2},{b:>2}): {why:38} ans = {ans}")
    return ans

trace_cases(*examples[2][:2])

xoá [-1,4)
  [-5,-4): rời nhau -> giữ nguyên                 ans = [[-5, -4]]
  [-3,-2): rời nhau -> giữ nguyên                 ans = [[-5, -4], [-3, -2]]
  [ 1, 2): giao -> bị xoá hết                     ans = [[-5, -4], [-3, -2]]
  [ 3, 5): giao -> mảnh phải [4,5)                ans = [[-5, -4], [-3, -2], [4, 5]]
  [ 8, 9): rời nhau -> giữ nguyên                 ans = [[-5, -4], [-3, -2], [4, 5], [8, 9]]


[[-5, -4], [-3, -2], [4, 5], [8, 9]]

In [15]:
def trace_bisect(intervals, toBeRemoved):
    x, y = toBeRemoved
    starts = [a for a, _ in intervals]; ends = [b for _, b in intervals]
    i = bisect.bisect_right(ends, x); j = bisect.bisect_left(starts, y)
    print(f"ends={ends}, starts={starts}")
    print(f"i = bisect_right(ends, {x}) = {i}  -> giữ nguyên {intervals[:i]}")
    print(f"j = bisect_left(starts, {y}) = {j} -> giữ nguyên {intervals[j:]}")
    print(f"bị ảnh hưởng: {intervals[i:j]}")
    res = remove_bisect(intervals, toBeRemoved)
    print("kết quả:", res)

trace_bisect(*examples[0][:2])

ends=[2, 4, 7], starts=[0, 3, 5]
i = bisect_right(ends, 1) = 0  -> giữ nguyên []
j = bisect_left(starts, 6) = 3 -> giữ nguyên []
bị ảnh hưởng: [[0, 2], [3, 4], [5, 7]]
kết quả: [[0, 1], [6, 7]]


## 9. Code nộp LeetCode

Đúng tên lớp/hàm/tham số theo doocs; dùng công thức `min`/`max` (Cách 3) cho gọn và ít lỗi dấu:

In [16]:
class Solution:
    def removeInterval(
        self, intervals: list[list[int]], toBeRemoved: list[int]
    ) -> list[list[int]]:
        x, y = toBeRemoved
        ans = []
        for a, b in intervals:
            if a < min(b, x):                 # mảnh trái [a, min(b, x))
                ans.append([a, min(b, x)])
            if max(a, y) < b:                 # mảnh phải [max(a, y), b)
                ans.append([max(a, y), b])
        return ans

for iv, rem, want in examples:
    got = Solution().removeInterval(iv, rem)
    print(got, "OK" if got == want else "SAI")

[[0, 1], [6, 7]] OK
[[0, 2], [3, 5]] OK
[[-5, -4], [-3, -2], [4, 5], [8, 9]] OK


## 10. Vì sao thuật toán đúng

**Bổ đề (hiệu hai khoảng).** Với `[a, b)` và `[x, y)` (`x < y`): `[a, b) \ [x, y) = [a, min(b, x)) ∪ [max(a, y), b)`.
Chứng minh: `t ∈ [a, b)` và `t ∉ [x, y)` ⇔ `a <= t < b` và (`t < x` hoặc `t >= y`) ⇔ (`a <= t < min(b, x)`) hoặc (`max(a, y) <= t < b`). Hai mảnh rời nhau vì mảnh trái `< x < y <=` mảnh phải.

**Tính đúng toàn cục.** `(∪ I_k) \ R = ∪ (I_k \ R)` (phép hiệu phân phối với phép hợp). Áp dụng bổ đề cho từng `I_k` ⇒ tập điểm đúng.

**Tính sắp xếp & rời nhau.** Các mảnh của `I_k` nằm trong `I_k`; các `I_k` rời nhau và tăng dần ⇒ mảnh của `I_k` đứng trước mảnh của `I_{k+1}`. Trong cùng `I_k`, mảnh trái trước mảnh phải. Mảnh rỗng bị loại nhờ điều kiện `<` chặt ⇒ kết quả hợp lệ. ∎

**Cách 2** chỉ là bổ đề viết dạng phân nhánh: khi rời nhau, một trong hai mảnh bằng cả khoảng và mảnh kia rỗng.

## 11. Phân tích Time / Space

Gọi `n = len(intervals)` (≤ 10⁴), `R` = độ dài miền toạ độ.

| Cách | Time | Space phụ (ngoài kết quả) |
|---|---|---|
| 1. Rời rạc hoá đơn vị | `O(R · n)` | `O(1)` |
| 2. Phân trường hợp | `O(n)` | `O(1)` |
| 3. Công thức min/max | `O(n)` | `O(1)` |
| 4. Bisect | `O(n)` (tìm vùng `O(log n)`) | `O(n)` cho `starts`/`ends` |
| 5. Sweep line | `O(n log n)` | `O(n)` |

Kết quả có tối đa `n + 1` khoảng (chỉ một khoảng có thể bị chẻ đôi) ⇒ đầu ra `O(n)`. Không có khác biệt average/worst cho Cách 2/3.

## 12. Bộ test

### 12.1 Ví dụ đề + edge case

So sánh chính xác với các lời giải giữ ranh giới (2, 3, 4, submit); với brute force và sweep (có gộp khoảng chạm nhau) thì so sau khi **chuẩn hoá**.

In [17]:
def normalize(ans):
    out = []
    for a, b in ans:
        if out and out[-1][1] == a:
            out[-1][1] = b
        else:
            out.append([a, b])
    return out

EXACT = {
    "cases": remove_cases,
    "cases_submit": lambda iv, r: SolutionCases().removeInterval(iv, r),
    "minmax": remove_minmax,
    "bisect": remove_bisect,
    "submit": lambda iv, r: Solution().removeInterval(iv, r),
}
MERGING = {"brute": remove_brute, "sweep": remove_sweep}

edge = [
    ([[0, 10]], [0, 10], []),                    # xoá đúng bằng khoảng
    ([[0, 10]], [-5, 15], []),                   # vùng xoá bao trùm
    ([[0, 10]], [10, 20], [[0, 10]]),            # chạm phải: không giao
    ([[0, 10]], [-5, 0], [[0, 10]]),             # chạm trái: không giao
    ([[0, 10]], [0, 3], [[3, 10]]),              # cắt sát đầu
    ([[0, 10]], [7, 10], [[0, 7]]),              # cắt sát cuối
    ([[0, 2], [2, 4]], [1, 3], [[0, 1], [3, 4]]),# hai khoảng chạm nhau
    ([[-10**9, 10**9]], [0, 1], [[-10**9, 0], [1, 10**9]]),
]
for name, f in EXACT.items():
    for iv, rem, want in examples + edge:
        snapshot = [p[:] for p in iv]
        got = f(iv, rem)
        assert got == want and is_valid(got), (name, iv, rem, got)
        assert iv == snapshot, (name, "đã sửa input")
for name, f in MERGING.items():
    for iv, rem, want in examples + edge:
        if name == "brute" and iv[-1][1] - iv[0][0] > 10**4:
            continue                              # brute quá chậm với toạ độ lớn
        assert f(iv, rem) == normalize(want), (name, iv, rem)
print("Mọi cách đúng với", len(examples + edge), "test")

Mọi cách đúng với 11 test


### 12.2 Đối chiếu ngẫu nhiên với brute force (seed cố định)

Sinh các khoảng rời nhau đã sắp xếp (có thể chạm nhau) trong `[-30, 30]`, vùng xoá ngẫu nhiên (kể cả nằm ngoài, bao trùm, chạm mép).

In [18]:
def random_case(rng):
    pts = sorted(rng.sample(range(-30, 31), 2 * rng.randint(1, 8)))
    iv = [[pts[2 * k], pts[2 * k + 1]] for k in range(len(pts) // 2)]
    if rng.random() < 0.3 and len(iv) >= 2:      # thỉnh thoảng cho hai khoảng chạm nhau
        k = rng.randrange(len(iv) - 1)
        iv[k][1] = min(iv[k][1], iv[k + 1][0])
        iv[k + 1][0] = iv[k][1]
        iv = [p for p in iv if p[0] < p[1]]
    x = rng.randint(-35, 34)
    y = rng.randint(x + 1, 35)
    return iv, [x, y]

rng = random.Random(1272)
for _ in range(2000):
    iv, rem = random_case(rng)
    want = remove_brute(iv, rem)
    for name, f in EXACT.items():
        got = f(iv, rem)
        assert is_valid(got) and normalize(got) == want, (name, iv, rem, got)
    assert remove_sweep(iv, rem) == want
    assert len({str(f(iv, rem)) for f in EXACT.values()}) == 1      # các cách chính xác trùng nhau
print("2000 test ngẫu nhiên: hợp lệ và khớp brute force")

2000 test ngẫu nhiên: hợp lệ và khớp brute force

### 12.3 Giới hạn `n = 10^4`, toạ độ `±10^9`

In [19]:
pts = sorted(random.Random(5).sample(range(-10**9, 10**9), 2 * 10**4))
big = [[pts[2 * k], pts[2 * k + 1]] for k in range(10**4)]
rem = [-3 * 10**8, 4 * 10**8]
ref = None
for name, f in list(EXACT.items()) + [("sweep", remove_sweep)]:
    t0 = time.perf_counter()
    res = f(big, rem)
    dt = time.perf_counter() - t0
    if name != "sweep":
        ref = res if ref is None else ref
        assert res == ref
    assert is_valid(res)
    print(f"{name:13}: {len(res)} khoảng, {dt:.4f}s")

cases        : 6504 khoảng, 0.0011s
cases_submit : 6504 khoảng, 0.0028s
minmax       : 6504 khoảng, 0.0019s
bisect       : 6504 khoảng, 0.0015s


submit       : 6504 khoảng, 0.0016s
sweep        : 6504 khoảng, 0.0047s


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Tư duy khoảng đóng khi cắt mảnh: `[a, x-1]` và `[y+1, b]`

Quen với khoảng đóng trên số nguyên nên "lùi 1 / tiến 1" ở chỗ cắt. Với khoảng nửa mở trên **số thực**, điểm `x` không bị xoá phần bên trái của nó: mảnh trái phải là `[a, x)`, không phải `[a, x−1)` — bản sai làm **mất** cả đoạn `[x−1, x)` và `[y, y+1)`.

In [20]:
def remove_wrong1(intervals, toBeRemoved):
    x, y = toBeRemoved
    ans = []
    for a, b in intervals:
        if a >= y or b <= x:
            ans.append([a, b])
        else:
            if a < x - 1:
                ans.append([a, x - 1])     # SAI: lùi 1 như khoảng đóng
            if b > y + 1:
                ans.append([y + 1, b])     # SAI: tiến 1
    return ans

for iv, rem, want in examples[:2]:
    print("Sai :", remove_wrong1(iv, rem), "| Đúng:", want)

Sai : [] | Đúng: [[0, 1], [6, 7]]
Sai : [[0, 1], [4, 5]] | Đúng: [[0, 2], [3, 5]]


**Sửa:** với `[a, b)` nửa mở, mảnh trái là `[a, x)`, mảnh phải là `[y, b)` — dùng đúng `x`, `y`.

### Lỗi 2 — Thêm mảnh mà không kiểm tra rỗng (sinh `[x, x]`)

Dùng công thức min/max nhưng quên điều kiện `a < min(b, x)` ⇒ sinh ra khoảng rỗng hoặc khoảng "âm".

In [21]:
def remove_wrong2(intervals, toBeRemoved):
    x, y = toBeRemoved
    ans = []
    for a, b in intervals:
        ans.append([a, min(b, x)])          # SAI: không kiểm tra rỗng
        ans.append([max(a, y), b])
    return ans

iv, rem, want = examples[0]
got = remove_wrong2(iv, rem)
print("Sai :", got, "| hợp lệ?", is_valid(got))
print("Đúng:", want)

Sai : [[0, 1], [6, 2], [3, 1], [6, 4], [5, 1], [6, 7]] | hợp lệ? False
Đúng: [[0, 1], [6, 7]]


### Lỗi 3 — Dùng `elif` cho hai mảnh (mất mảnh phải khi bị chẻ đôi)

Khi khoảng **bao trùm** vùng xoá, cần thêm **cả hai** mảnh. `if … elif …` chỉ thêm một.

In [22]:
def remove_wrong3(intervals, toBeRemoved):
    x, y = toBeRemoved
    ans = []
    for a, b in intervals:
        if a >= y or b <= x:
            ans.append([a, b])
        elif a < x:                          # SAI: elif thay vì if
            ans.append([a, x])
        elif b > y:
            ans.append([y, b])
    return ans

iv, rem, want = examples[1]
print("Sai :", remove_wrong3(iv, rem))
print("Đúng:", want)

Sai : [[0, 2]]
Đúng: [[0, 2], [3, 5]]


### Lỗi 4 — Chỉ xử lý "khoảng đầu tiên giao" rồi `break`

Nghĩ rằng vùng xoá chỉ chạm một khoảng; sau khi cắt khoảng đầu tiên giao nhau thì dừng/copy phần còn lại nguyên vẹn — sai khi vùng xoá trải qua nhiều khoảng.

In [23]:
def remove_wrong4(intervals, toBeRemoved):
    x, y = toBeRemoved
    ans = []
    for k, (a, b) in enumerate(intervals):
        if a >= y or b <= x:
            ans.append([a, b])
            continue
        if a < x:
            ans.append([a, x])
        if b > y:
            ans.append([y, b])
        ans.extend(p[:] for p in intervals[k + 1:])   # SAI: giả định chỉ 1 khoảng bị ảnh hưởng
        break
    return ans

iv, rem, want = examples[0]
print("Sai :", remove_wrong4(iv, rem))
print("Đúng:", want)

Sai : [[0, 1], [3, 4], [5, 7]]
Đúng: [[0, 1], [6, 7]]


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **Bài khoảng (interval)**: xác định ngay là **đóng** hay **nửa mở**; viết điều kiện giao `max(a, c) < min(b, d)` (nửa mở) hoặc `<=` (đóng).
- **Hiệu của hai khoảng** = tối đa 2 mảnh: `[a, min(b, x))` và `[max(a, y), b)` — nhớ công thức này thay vì liệt kê 5 trường hợp.
- Input đã **sắp xếp + rời nhau** ⇒ xử lý từng khoảng độc lập, output tự động sắp xếp; có thể `bisect` để tìm vùng ảnh hưởng.
- Luôn có test "chạm mép" (`b == x`, `a == y`) và "bao trùm" (khoảng chứa vùng xoá, vùng xoá chứa khoảng).
- Viết sẵn `is_valid` + brute force theo đơn vị nguyên để kiểm tra nhanh.

## 15. Biến thể của bài

1. **Xoá nhiều vùng** `toBeRemoved` (đã sắp) ⇒ hai con trỏ quét song song, `O(n + m)`.
2. **LeetCode 57 — Insert Interval**: chèn và gộp (ngược với xoá).
3. **LeetCode 986 — Interval List Intersections**: giao hai danh sách khoảng bằng hai con trỏ.
4. **LeetCode 715 — Range Module**: thêm / xoá / truy vấn khoảng liên tục (cấu trúc dữ liệu động).
5. **Tổng độ dài còn lại** sau khi xoá: `Σ(b − a)` của kết quả, hoặc tổng ban đầu trừ độ dài giao.

Minh hoạ biến thể 1 và 5:

In [24]:
def remove_many(intervals, removes):
    """removes: danh sách vùng xoá đã sắp, rời nhau."""
    ans = []
    j = 0
    for a, b in intervals:
        cur = a
        while j < len(removes) and removes[j][1] <= cur:
            j += 1                              # bỏ vùng xoá nằm hẳn bên trái
        k = j
        while k < len(removes) and removes[k][0] < b:
            x, y = removes[k]
            if cur < x:
                ans.append([cur, x])
            cur = max(cur, y)
            k += 1
        if cur < b:
            ans.append([cur, b])
    return ans

def remaining_length(intervals, toBeRemoved):
    x, y = toBeRemoved
    total = sum(b - a for a, b in intervals)
    cut = sum(max(0, min(b, y) - max(a, x)) for a, b in intervals)
    return total - cut

print(remove_many([[0, 10], [12, 20]], [[1, 2], [4, 5], [9, 13], [15, 16]]))
iv, rem, want = examples[2]
print(remaining_length(iv, rem), "==", sum(b - a for a, b in want))

[[0, 1], [2, 4], [5, 9], [13, 15], [16, 20]]
4 == 4


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Rời rạc hoá đơn vị | Xét từng ô `[t, t+1)` rồi gộp | `O(R·n)` | `O(1)` | Oracle khi toạ độ nhỏ |
| 2. Phân trường hợp | Rời nhau / có giao → mảnh trái, mảnh phải | `O(n)` | `O(1)` | Lời giải doocs, dễ giải thích |
| 3. Công thức min/max | Hiệu = `[a,min(b,x)) ∪ [max(a,y),b)` | `O(n)` | `O(1)` | Ngắn, ít lỗi dấu |
| 4. Bisect | Tìm vùng ảnh hưởng `[i, j)` | `O(n)` (tìm `O(log n)`) | `O(n)` | Danh sách lớn, sửa tại chỗ |
| 5. Sweep line | Quét sự kiện với 2 bộ đếm | `O(n log n)` | `O(n)` | Nhiều phép toán tập hợp trên khoảng |

## 17. Bài tập vận dụng

### Bài 1 (đoán output)
`Solution().removeInterval([[1,3],[4,6],[7,9]], [2,8])` = ?

### Bài 2 (tính tay)
Với `intervals = [[-4,-1],[0,2],[5,9]]` và `toBeRemoved = [-2,6]`, phân loại từng khoảng (rời / bị phủ / cắt trái / cắt phải / chẻ đôi) và viết kết quả.

### Bài 3 (điền TODO)
Hoàn thiện `removed_part(intervals, toBeRemoved)`: trả về danh sách các khoảng **bị xoá thực sự** (phần giao), đã sắp xếp.

### Bài 4 (tự viết lại)
Viết lại lời giải dạng **sửa tại chỗ**: `remove_inplace(intervals, toBeRemoved)` thay đổi `intervals` (dùng bisect để tìm `[i, j)` rồi gán lát cắt `intervals[i:j] = middle`) và trả về chính danh sách đó.

### Bài 5 (mini challenge)
Viết `remove_closed(intervals, toBeRemoved)` cho phiên bản **khoảng đóng trên số nguyên**: mỗi `[a, b]` là tập số nguyên `a..b` (cả hai đầu), xoá tập số nguyên `x..y`.

In [25]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def removed_part(intervals, toBeRemoved):
    # TODO: với mỗi [a, b), giao = [max(a, x), min(b, y)); thêm nếu không rỗng
    pass

# Bài 4 — TODO
def remove_inplace(intervals, toBeRemoved):
    # TODO: i = bisect_right(ends, x), j = bisect_left(starts, y), intervals[i:j] = middle
    pass

# Bài 5 — TODO
def remove_closed(intervals, toBeRemoved):
    # TODO: mảnh trái [a, min(b, x-1)], mảnh phải [max(a, y+1), b] — nhớ điều kiện không rỗng dùng <=
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) `[1,3)` bị cắt phải ⇒ `[1,2)`. (2) `[4,6)` bị phủ hết. (3) `[7,9)` bị cắt trái ⇒ `[8,9)` ⇒ `[[1,2],[8,9]]`.

**Bài 2.** (1) `[-4,-1)`: `a=-4 < -2` và `b=-1 > -2` ⇒ cắt phải ⇒ `[-4,-2)`. (2) `[0,2)`: bị phủ. (3) `[5,9)`: cắt trái ⇒ `[6,9)`.

**Bài 3.** (1) Giao `[max(a, x), min(b, y))`. (2) Chỉ thêm khi `max < min`. (3) Tổng độ dài phần xoá + phần còn lại = tổng ban đầu — dùng để kiểm tra.

**Bài 4.** (1) Tính `middle` như Cách 4. (2) Gán lát cắt thay thế cả đoạn `[i, j)` (kể cả khi `i == j`: chèn rỗng — không đổi). (3) So với `Solution` trên test ngẫu nhiên (truyền bản sao).

**Bài 5.** (1) Khoảng đóng số nguyên `[a, b]` ⇔ nửa mở `[a, b+1)`. (2) Có thể chuyển đổi, gọi `Solution`, rồi chuyển ngược `[p, q) → [p, q−1]`. (3) Hoặc viết trực tiếp với điều kiện `<=`.

## 19. Đáp án tham khảo

In [26]:
# Bài 1
assert Solution().removeInterval([[1, 3], [4, 6], [7, 9]], [2, 8]) == [[1, 2], [8, 9]]
# Bài 2
assert Solution().removeInterval([[-4, -1], [0, 2], [5, 9]], [-2, 6]) == [[-4, -2], [6, 9]]

# Bài 3
def removed_part_sol(intervals, toBeRemoved):
    x, y = toBeRemoved
    out = []
    for a, b in intervals:
        lo, hi = max(a, x), min(b, y)
        if lo < hi:
            out.append([lo, hi])
    return out

# Bài 4
def remove_inplace_sol(intervals, toBeRemoved):
    x, y = toBeRemoved
    ends = [b for _, b in intervals]
    starts = [a for a, _ in intervals]
    i = bisect.bisect_right(ends, x)
    j = bisect.bisect_left(starts, y)
    middle = []
    if i < j:
        if intervals[i][0] < x:
            middle.append([intervals[i][0], x])
        if intervals[j - 1][1] > y:
            middle.append([y, intervals[j - 1][1]])
    intervals[i:j] = middle
    return intervals

# Bài 5
def remove_closed_sol(intervals, toBeRemoved):
    x, y = toBeRemoved
    ans = []
    for a, b in intervals:
        if a <= min(b, x - 1):
            ans.append([a, min(b, x - 1)])
        if max(a, y + 1) <= b:
            ans.append([max(a, y + 1), b])
    return ans

def remove_closed_brute(intervals, toBeRemoved):
    x, y = toBeRemoved
    pts = sorted({t for a, b in intervals for t in range(a, b + 1)} - set(range(x, y + 1)))
    out = []
    for t in pts:
        if out and out[-1][1] == t - 1:
            out[-1][1] = t
        else:
            out.append([t, t])
    return out

rng = random.Random(72)
for _ in range(1000):
    iv, rem = random_case(rng)
    total = sum(b - a for a, b in iv)
    rp = removed_part_sol(iv, rem)
    assert is_valid(rp)
    assert total == sum(b - a for a, b in rp) + sum(b - a for a, b in Solution().removeInterval(iv, rem))
    assert remove_inplace_sol([p[:] for p in iv], rem) == Solution().removeInterval(iv, rem)
    # khoảng đóng số nguyên: dùng các khoảng không chạm nhau để brute force so sánh sau khi gộp
    closed = [[a, b - 1] for a, b in iv]
    got = remove_closed_sol(closed, [rem[0], rem[1] - 1])
    merged = []
    for a, b in got:
        if merged and merged[-1][1] == a - 1:
            merged[-1][1] = b
        else:
            merged.append([a, b])
    assert merged == remove_closed_brute(closed, [rem[0], rem[1] - 1])
print("Đáp án Bài 1–5: mọi assert đều qua")

Đáp án Bài 1–5: mọi assert đều qua


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template hiệu khoảng nửa mở:**

```python
x, y = toBeRemoved
ans = []
for a, b in intervals:
    if a < min(b, x):  ans.append([a, min(b, x)])     # mảnh trái
    if max(a, y) < b:  ans.append([max(a, y), b])     # mảnh phải
```

**Template phân trường hợp (doocs):** rời nhau (`a >= y or b <= x`) ⇒ giữ; ngược lại `if a < x` thêm `[a, x]`, `if b > y` thêm `[y, b]` (hai `if` độc lập).

**Checklist trước khi submit:**
- [ ] Khoảng **nửa mở**: rời nhau khi `b <= x` hoặc `a >= y`.
- [ ] Không sinh khoảng rỗng (điều kiện `<` chặt).
- [ ] Trường hợp bao trùm: thêm **cả hai** mảnh (`if`, không `elif`).
- [ ] Không `break` sau khoảng giao đầu tiên — vùng xoá có thể trải nhiều khoảng.
- [ ] Test chạm mép, bao trùm, xoá hết, xoá không gì.
- [ ] Đúng chữ ký: `removeInterval(self, intervals, toBeRemoved) -> list[list[int]]`.